In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/customer_churn_cleaned.csv")

In [2]:
X = df.drop(columns=["churn_flag"])
y = df["churn_flag"]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
import joblib

preprocessor = joblib.load("../models/preprocessor.pkl")

In [6]:
%pip install imbalanced-learn


   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.4/101.7 MB 16.8 MB/s eta 0:00:06
   -- ------------------------------------- 6.8/101.7 MB 18.3 MB/s eta 0:00:06
   ---- ----------------------------------- 11.3/101.7 MB 19.6 MB/s eta 0:00:05
   ----- ---------------------------------- 14.4/101.7 MB 18.5 MB/s eta 0:00:05
   ------ --------------------------------- 17.3/101.7 MB 17.3 MB/s eta 0:00:05
   -------- ------------------------------- 21.0/101.7 MB 17.4 MB/s eta 0:00:05
   --------- ------------------------------ 24.9/101.7 MB 17.3 MB/s eta 0:00:05
   ----------- ---------------------------- 28.6/101.7 MB 17.4 MB/s eta 0:00:05
   ------------ --------------------------- 31.7/101.7 MB 17.2 MB/s eta 0:00:05
   ------------- -------------------------- 35.1/101.7 MB 16.9 MB/s eta 0:00:04
   --------------- ------------------------ 38.8/101.7 MB 17

In [11]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

logistic_model = ImbPipeline(
    steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", LogisticRegression(max_iter=1000))
    ]
)

rf_model = ImbPipeline(
    steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
    ]
)

xgb_model = ImbPipeline(
    steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, eval_metric="logloss"))
    ]
)

In [12]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

In [13]:
results = []
trained_models = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    trained_models[name] = model

Training: Logistic Regression
Training: Random Forest
Training: XGBoost


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_probability = (model.predict_proba(X_test)[:,1])
    results = {
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_probability)
    }
    return results

In [23]:
lr_result = evaluate_model("Logistic Regression", logistic_model, X_test, y_test)

rf_result = evaluate_model("Random Forest", rf_model, X_test, y_test)

xgb_result = evaluate_model("XGBoost", xgb_model, X_test, y_test)

results = [lr_result, rf_result, xgb_result]

In [25]:
comparison = pd.DataFrame(results)
comparison

,Model,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.373472,0.749164,0.498455,0.677560
1,Random Forest,0.523504,0.182088,0.270196,0.673098
2,XGBoost,0.663327,0.123003,0.207524,0.675900


In [26]:
#Best model

comparison.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.373472,0.749164,0.498455,0.677560
1,Random Forest,0.523504,0.182088,0.270196,0.673098
2,XGBoost,0.663327,0.123003,0.207524,0.675900


In [ ]:
#Final evaulation with best model

In [28]:
best_model = trained_models["Logistic Regression"]

In [29]:
final_predictions = best_model.predict(X_test)

In [31]:
from sklearn.metrics import classification_report

print(classification_report(y_test, final_predictions))

              precision    recall  f1-score   support

           0       0.85      0.54      0.66      7309
           1       0.37      0.75      0.50      2691

    accuracy                           0.59     10000
   macro avg       0.61      0.64      0.58     10000
weighted avg       0.72      0.59      0.62     10000



In [32]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, final_predictions)

array([[3927, 3382],
       [ 675, 2016]])

In [33]:
#Save model

import joblib

joblib.dump(best_model, "../models/best_churn_model.pkl")

['../models/best_churn_model.pkl']